# OpenMHC Quickstart

End-to-end walkthrough of the public evaluation API:

1. Download the tiny dataset
2. Define a trivial `Imputer` (mean imputation)
3. Run `evaluate_imputation` and inspect the results
4. Define a trivial `Encoder` (mean-pool over the sensor channels)
5. Run `evaluate_downstream` and inspect the results

This notebook is the smallest end-to-end loop. It runs on CPU and finishes in a few minutes against the tiny version.

## 1. Install + download the dataset

If you haven't already:

```bash
pip install -e ..
```

Then download the tiny dataset (~TBD MB).

In [ ]:
import openmhc

# First time only: downloads to ~/.cache/openmhc/data unless MHC_DATA_DIR is set
data_path = openmhc.download_dataset(version="tiny")
print(f"dataset is at: {data_path}")

## 2. Define a trivial Imputer

Models implement the `Imputer` protocol — two methods, no inheritance:
- `fit(data, masks)` — see real training data once
- `impute(data, observed_mask, target_mask)` — fill in artificially-masked positions

We'll just predict the per-channel global mean.

In [ ]:
import numpy as np

class MeanImputer:
    """Predict the channel-wise mean for every masked position."""

    def fit(self, data: np.ndarray, masks: np.ndarray) -> None:
        # data: (N, 19, 1440) with NaN at missing. Mean over (samples, time).
        self.means = np.nanmean(data, axis=(0, 2))

    def impute(
        self,
        data: np.ndarray,
        observed_mask: np.ndarray,
        target_mask: np.ndarray,
    ) -> np.ndarray:
        out = data.copy()
        for ch in range(19):
            target = target_mask[:, ch, :] == 1
            out[:, ch, :][target] = self.means[ch]
        return out.astype(np.float32)

## 3. Run imputation evaluation

`evaluate_imputation` runs your imputer against all 6 masking scenarios (random noise, temporal slices, sleep gaps, etc.) on the test split.

In [ ]:
results = openmhc.evaluate_imputation(MeanImputer())
results.summary()

## 4. Define a trivial Encoder

For Track 1 (outcome prediction) the protocol is just `encode(weekly_tensors) -> embeddings`. Here we mean-pool over the time axis.

In [ ]:
class MeanPoolEncoder:
    """Average over the 168 hours, take the 19 sensor channels."""

    def encode(self, weekly_tensors: np.ndarray) -> np.ndarray:
        # weekly_tensors: (B, 168, 38). Cols 0-18 sensors, 19-37 missingness.
        return weekly_tensors[:, :, :19].mean(axis=1).astype(np.float32)

## 5. Run downstream evaluation

`evaluate_downstream` extracts your embeddings, then fits a linear probe per task (logistic regression for binary, ordinal logistic for ordinal, ridge regression for continuous).

In [ ]:
# tasks="all" is fine on the tiny dataset; restrict if you want a faster loop
results = openmhc.evaluate_downstream(MeanPoolEncoder(), tasks=["BiologicalSex", "age"])
results.summary()

## 6. Generate a leaderboard submission

`results.to_submission_yaml(...)` produces a paste-ready body matching the textareas in the [submission issue template](https://github.com/AshleyLab/myheartcounts-dataset/issues/new?template=submission.yml).

For Track 2 imputation, skill scores and per-category subgroup scores are computed locally against the frozen LOCF baseline (`data/baselines/imputation_locf.json`). For Track 1, those fields are emitted as `—` and filled in by the maintainers from `raw_metrics` during ingestion.

In [ ]:
imputation_results = openmhc.evaluate_imputation(MeanImputer())

submission_body = imputation_results.to_submission_yaml(
    method_name="MeanImputer (demo)",
    submitter_team="Stanford CS",
    paper_url="https://arxiv.org/abs/0000.00000",
    code_url="https://github.com/example/mean-imputer",
    method_category="Statistical / Classical baseline",
)
print(submission_body)

Once you have a body that looks right:

1. Open a [submission issue](https://github.com/AshleyLab/myheartcounts-dataset/issues/new?template=submission.yml).
2. Fill in the single-input fields (method name, paper URL, …) and paste the three textarea blocks above.
3. The HuggingFace Space ingests merged submissions and the public leaderboard at `https://myheartcounts.stanford.edu/benchmark` rebuilds automatically.

Submissions must follow the standard protocol — same split file, masking config, label-validity criterion as the paper. The submission template enforces required fields.